# Structured and Constrained LLM Output with Ludwig

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/llm_structured_output/structured_output.ipynb)

Large language models are trained to produce fluent text, but they have no built-in guarantee that their output follows a particular format. **Constrained decoding** solves this by restricting token sampling at inference time so the model can only ever produce output that satisfies a given constraint.

**This notebook covers:**
1. Entity extraction with a JSON schema constraint
2. Sentiment classification with a regex constraint (guaranteed to produce only `positive`, `negative`, or `neutral`)
3. Logits extraction from LLM output
4. Side-by-side comparison: constrained vs unconstrained decoding

All examples use small, freely available models that run on a free Colab T4 GPU.

In [ ]:
!pip install "ludwig[llm]" --quiet

## Setup

In [ ]:
import json
import textwrap

import pandas as pd
import torch
import yaml

from ludwig.api import LudwigModel

# Check GPU availability
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {device_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("No GPU detected. Running on CPU — inference will be slower.")

## Entity extraction with JSON schema

We configure Ludwig to constrain the LLM's output to a specific JSON schema. The schema describes the shape of the expected output — Ludwig compiles it into logit masks so only valid JSON tokens can be sampled.

The model is `microsoft/phi-2` (2.7 B parameters, fits comfortably on a T4 with 4-bit quantization).

In [ ]:
entity_config = yaml.safe_load("""
model_type: llm
base_model: microsoft/phi-2

quantization:
  bits: 4

prompt:
  task: >
    Extract the named entities from the input text and return them as a JSON
    object with this structure:
    {"entities": [{"text": "...", "type": "PERSON|ORG|LOC|DATE"}]}.
    Return only valid JSON, nothing else.

input_features:
  - name: text
    type: text

output_features:
  - name: output
    type: text
    decoder:
      type: text_parser
      json_schema:
        type: object
        properties:
          entities:
            type: array
            items:
              type: object
              properties:
                text:
                  type: string
                type:
                  type: string
                  enum: [PERSON, ORG, LOC, DATE]
              required: [text, type]
        required: [entities]
        additionalProperties: false

generation:
  max_new_tokens: 200
  temperature: 0.1
  do_sample: false

backend:
  type: local
""")

entity_samples = [
    "Apple Inc. was founded by Steve Jobs in Cupertino, California on April 1, 1976.",
    "Elon Musk announced that Tesla will open a new Gigafactory in Berlin next year.",
    "The United Nations headquarters is located in New York City.",
]

entity_df = pd.DataFrame({"text": entity_samples})
entity_df

In [ ]:
entity_model = LudwigModel(config=entity_config)
entity_preds, _, _ = entity_model.predict(dataset=entity_df)

for text, pred in zip(entity_samples, entity_preds["output_predictions"]):
    print(f"Input:  {text}")
    try:
        parsed = json.loads(pred)
        entities = parsed.get("entities", [])
        print(f"Output: {len(entities)} entities")
        for ent in entities:
            print(f"  - '{ent['text']}' ({ent['type']})")
    except json.JSONDecodeError:
        print(f"Output (raw): {pred}")
    print()

The output is guaranteed to be valid JSON matching the schema. The LLM cannot produce malformed JSON, extra prose, or entity types outside the allowed enum.

## Classification with constrained tokens

For classification tasks we can constrain the output to a regex that only allows the valid class labels. Here we restrict the model to `positive`, `negative`, or `neutral`.

We use `Qwen/Qwen2-0.5B-Instruct` (0.5 B parameters) — small enough to run quickly even on CPU.

In [ ]:
sentiment_constrained_config = yaml.safe_load("""
model_type: llm
base_model: Qwen/Qwen2-0.5B-Instruct

prompt:
  task: >
    Classify the sentiment of the following text.
    Respond with exactly one word: positive, negative, or neutral.

input_features:
  - name: text
    type: text

output_features:
  - name: sentiment
    type: text
    decoder:
      type: text_parser
      # Regex constraint — only these three tokens can be emitted.
      regex: "(positive|negative|neutral)"

generation:
  max_new_tokens: 10
  temperature: 0.0
  do_sample: false

backend:
  type: local
""")

sentiment_samples = [
    "I absolutely loved this product! It exceeded all my expectations.",
    "The service was terrible and the food was cold.",
    "The movie was okay, nothing special.",
    "This is the best laptop I have ever owned. Highly recommend.",
    "I waited two hours and they still got my order wrong.",
    "The weather today is neither good nor bad.",
]

sentiment_df = pd.DataFrame({"text": sentiment_samples})

sentiment_model = LudwigModel(config=sentiment_constrained_config)
sentiment_preds, _, _ = sentiment_model.predict(dataset=sentiment_df)

for text, label in zip(sentiment_samples, sentiment_preds["sentiment_predictions"]):
    print(f"{str(label):<10}  {text}")

Every prediction is one of the three valid labels. No post-processing or error handling is required.

## Logits extraction

Ludwig can return the raw logits (pre-softmax scores over the vocabulary) for each generated token. This is useful for:

- Computing token-level confidence scores
- Calibration and uncertainty estimation
- Analysing what the model "considered" at each step
- Downstream ensemble or reranking tasks

In [ ]:
logits_config = yaml.safe_load("""
model_type: llm
base_model: Qwen/Qwen2-0.5B-Instruct

prompt:
  task: "Answer with a single word."

input_features:
  - name: text
    type: text

output_features:
  - name: response
    type: text
    # Request logits to be returned alongside the prediction.
    output_logits: true

generation:
  max_new_tokens: 5
  temperature: 0.0
  do_sample: false

backend:
  type: local
""")

logits_model = LudwigModel(config=logits_config)
logits_df = pd.DataFrame({"text": ["Is Python a programming language?"]})

logits_preds, output_df, _ = logits_model.predict(dataset=logits_df, collect_predictions=True)

print("Prediction:", logits_preds["response_predictions"].iloc[0])

if "response_logits" in output_df.columns:
    import numpy as np
    logits = output_df["response_logits"].iloc[0]
    logits_arr = np.array(logits)
    print(f"Logits shape: {logits_arr.shape}")
    # Convert to probabilities for the first generated token
    probs = np.exp(logits_arr[0]) / np.exp(logits_arr[0]).sum()
    top5_idx = np.argsort(probs)[::-1][:5]
    print("Top-5 token probabilities (first generated token):")
    for idx in top5_idx:
        print(f"  token {idx}: {probs[idx]:.4f}")
else:
    print("Logits column not present in output.")
    print("Available columns:", list(output_df.columns))

## Comparison: constrained vs unconstrained

The following cell runs the same sentiment classification task with and without the regex constraint, then prints both outputs side by side to show how constrained decoding eliminates invalid responses.

In [ ]:
# Unconstrained config — same prompt, no decoder constraint
sentiment_unconstrained_config = yaml.safe_load("""
model_type: llm
base_model: Qwen/Qwen2-0.5B-Instruct

prompt:
  task: >
    Classify the sentiment of the following text.
    Respond with exactly one word: positive, negative, or neutral.

input_features:
  - name: text
    type: text

output_features:
  - name: sentiment
    type: text

generation:
  max_new_tokens: 30
  temperature: 0.7

backend:
  type: local
""")

unconstrained_model = LudwigModel(config=sentiment_unconstrained_config)
unconstrained_preds, _, _ = unconstrained_model.predict(dataset=sentiment_df)

# The constrained model was already run above
constrained_labels = sentiment_preds["sentiment_predictions"].tolist()
unconstrained_labels = unconstrained_preds["sentiment_predictions"].tolist()

valid_labels = {"positive", "negative", "neutral"}

print(f"{'Input':<48} {'Unconstrained':<32} {'Constrained'}")
print("-" * 100)
for text, unc, con in zip(sentiment_samples, unconstrained_labels, constrained_labels):
    short = textwrap.shorten(text, width=46)
    unc_str = str(unc).strip()
    # Highlight invalid outputs
    flag = "  *** INVALID" if unc_str.lower() not in valid_labels else ""
    print(f"{short:<48} {unc_str:<32} {str(con)}{flag}")

n_invalid = sum(1 for u in unconstrained_labels if str(u).strip().lower() not in valid_labels)
print(f"\nUnconstrained — invalid outputs: {n_invalid}/{len(sentiment_samples)}")
print(f"Constrained   — invalid outputs: 0/{len(sentiment_samples)} (guaranteed)")